# 6 · The reparameterized 32-16, trained on the stream

The first strategy sweep of `04_upscale_strategies.ipynb` - on drawn desktops, before the
stream-degraded data existed - picked the **reparameterized 32-16**. While training, each
convolution is three parallel branches; afterwards they are merged into one, so it ships
as notebook 3's plain 32-16 graph. This notebook trains it on the patches from
`00_data_preprocess.ipynb` - photos and video sent through the stream's H.264 - and
publishes it at the tile a client runs.

Notebook 4, run again on this data, finds reparameterization worth nothing measurable, so
this is not the model to ship; re-pointing this notebook at one of notebook 4's two tiers
is on the list in `summary.md`, "Next".

Steps:

1. **Data** - load the patch pairs, and label every validation patch by bitrate and content.
2. **Model** - the reparameterized 32-16; it must start as bicubic.
3. **Train** - one run, until it has converged.
4. **Results** - against bilinear, bicubic and Lanczos-3, overall and by bitrate and content.
5. **Save** - merge the branches, write a checkpoint, check it loads back the same, run a
   whole frame.
6. **Export** - float32, float16 and a whole frame at the client's tile, with bilinear and
   bicubic beside it.

Needs: `00_data_preprocess.ipynb`. Results: `summary.md`, "Notebooks".
Writes `checkpoints/upscale_reparam.pt`.

In [ ]:
"""Settings."""

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn.functional as F
from PIL import Image
from torch.utils.data import DataLoader, Subset

from helpers import degrade, metrics, models, patches, sources, training, webexport
from helpers.training import DATA_DIR, DEVICE

CHECKPOINT = training.CHECKPOINT_DIR / "upscale_reparam.pt"        # the finished, merged model
TRAINING = training.CHECKPOINT_DIR / "reparam_runs" / "seed0.pt"   # the training, saved every epoch

# The network (see "2. Model")
WIDTHS = (32, 16)     # channels of the body's convolutions
STEM = 5              # kernel size of the first convolution; the ones after it are 3x3
EXPAND = 4            # how much wider the extra training branch is
MAX_EPOCHS = 500      # a safety limit; training stops once it has converged
BATCH_SIZE = 32
LEARNING_RATE = 1e-4  # not 1e-3, where the model never beats bicubic (see "3. Train")
SEED = 0

STEP_WIDTH, STEP_HEIGHT = 320, 180       # LR pixels one client tile adds to the picture (notebook 5)
FRAME_WIDTH, FRAME_HEIGHT = 1920, 1080   # the whole-frame graph
NAME = "reparam_32-16"

training.seed_everything(SEED)

## 1. Data

`train` is what the model learns from, with random flips and rotations as free extra
examples. `val` is held out: every score below is measured on it.

The LR patches carry a halo of extra pixels, so the model's output is a bit too large;
`trim` cuts that border off before the loss and the scores.

In [ ]:
manifest = training.load_manifest()
training.describe(manifest)
SCALE = manifest["scale"]
train_loader, val_loader = training.loaders(DATA_DIR, manifest, BATCH_SIZE, val_batch_size=128)
trim = training.trimmer(manifest)

lr_batch, hr_batch = next(iter(train_loader))
print("lr batch", tuple(lr_batch.shape), " hr batch", tuple(hr_batch.shape))
training.show_patches({"LR (with halo)": lr_batch[:4], "HR": hr_batch[:4]})

### Label every validation patch

The manifest records how each source was degraded, so every validation patch also gets
two labels, used to split the scores in "4. Results": its **bitrate** (how hard the codec
squeezed it) and its **content** (video, or an image that stands still, slides or zooms).

In [ ]:
def bitrate_group(bpp):
    """bpp = bits per pixel per frame, None = no codec. Split at the host's default bitrate,
    the middle of the range drawn."""
    if bpp is None:
        return "no codec"
    return "bpp 0.46-1.0" if bpp >= degrade.STREAM_BPP else "bpp 0.2-0.46"


def content_group(source):
    """A video, or an image and how the window over it moved (still, slide or zoom)."""
    return f"image, {source['motion']['kind']}" if "motion" in source else "video"


members = {   # way of grouping -> group -> the indices of its validation patches
    "bitrate": {group: [] for group in ("no codec", "bpp 0.46-1.0", "bpp 0.2-0.46")},
    "content": {group: [] for group in ("image, still", "image, slide", "image, zoom", "video")},
}
for source in manifest["splits"]["val"]["sources"]:
    first, end = source["patches"]   # this source's patches are first, first + 1, ..., end - 1
    members["bitrate"][bitrate_group(source["degradation"]["bpp"])].extend(range(first, end))
    members["content"][content_group(source)].extend(range(first, end))

for key, groups in members.items():
    print(f"{key:<8}" + ", ".join(f"{group}: {len(indices)}" for group, indices in groups.items()))

## 2. Model

    LR --> bicubic skip (fixed 5x5 conv) -------------------------------------------+
    LR --> RepConv 5x5 -> ReLU -> RepConv 3x3 -> ReLU -> output conv 3x3 ------------+--> add --> PixelShuffle --> HR

- **Three branches while training.** Each `models.RepConv` is a k x k, a 1x1, and a 4x
  wider 1x1 followed by a k x k. All three are linear, so after training they merge into
  one k x k with the same output: more to learn with, notebook 3's graph to run.
- **Starts as bicubic.** The output convolution starts at zero, as in notebook 3.

**Speed** is notebook 3's: merged, this is the same graph, so it is not measured again.

In [ ]:
models.check_skip_is_bicubic(SCALE)
model = models.ReparamUpscaler(WIDTHS, STEM, EXPAND, scale=SCALE).to(DEVICE)
assert model.halo <= manifest["halo"], "the model reaches further than the patches' halo"
print(model)
print(f"{models.count_parameters(model, learned_only=True):,} learned parameters while training, "
      f"{models.count_parameters(models.PlainUpscaler(WIDTHS, STEM), learned_only=True):,} after merging, "
      f"halo {model.halo}")

In [ ]:
"""Before training: the model must score exactly like bicubic."""

bicubic = models.Resize(SCALE).to(DEVICE)
baseline = training.evaluate(bicubic, val_loader, trim)
start = training.evaluate(model, val_loader, trim)
metrics.print_table({"bicubic": baseline, "untrained model": start})
assert abs(start["wpsnr"] - baseline["wpsnr"]) < 1e-3, "the model should start as bicubic"

## 3. Train

Adam, batch 32, and a loss that matches the score: L1 with brightness weighted 6:1:1
(`metrics.luma_chroma_l1`). After every epoch the model is scored on the validation
patches by `wpsnr`, and the best epoch is kept. Training runs until the score stops
improving, halving the learning rate at every plateau; `MAX_EPOCHS` is only a safety limit.

- **Learning rate 1e-4.** Three branches moving at once take a larger step on the merged
  kernel than one convolution does. At 1e-3 the model switches off most of its narrow
  channels and stays at bicubic (notebook 4); at 1e-4 it is ahead within a few epochs.
- **One run.** How far apart two seeds land is measured in notebook 4: 0.11 dB wpsnr for
  the 32-16. A smaller difference is luck.
- **Saved every epoch** to `checkpoints/reparam_runs/`: run the cell again after an
  interruption and it carries on, and a finished training just loads.

The best epoch is picked on the validation patches, and the same patches score the model
below, so that score is slightly optimistic.

In [ ]:
history, best = training.train(model, train_loader, val_loader, trim, MAX_EPOCHS, learning_rate=LEARNING_RATE,
                               loss_fn=metrics.luma_chroma_l1, checkpoint=TRAINING)
training.plot_history(history, baseline, loss_name="brightness-weighted L1")

## 4. Results

Against bilinear, bicubic and Lanczos-3 on the same validation patches. Lanczos-3 is
Pillow's: ONNX cannot express it, so it shows the best a plain filter can do.

Split by group: a gain only in the "no codec" group means the model learned to sharpen; a
gain at low bitrates means it learned to clean up the codec.

In [ ]:
class Lanczos3(torch.nn.Module):
    """Pillow's Lanczos-3 as a module (per patch, on the CPU), so it can be scored."""

    def forward(self, x):
        size = (x.shape[3] * SCALE, x.shape[2] * SCALE)   # Pillow wants (width, height)
        images = (x.clamp(0, 1) * 255).round().byte().permute(0, 2, 3, 1).cpu().numpy()
        upscaled = [np.asarray(Image.fromarray(image).resize(size, Image.LANCZOS)) for image in images]
        return torch.from_numpy(np.stack(upscaled)).permute(0, 3, 1, 2).float().div(255).to(x.device)


methods = {"bilinear": models.Resize(SCALE, "bilinear").to(DEVICE), "bicubic": bicubic,
           "lanczos3": Lanczos3(), NAME: model}
results = {name: training.evaluate(method, val_loader, trim) for name, method in methods.items()}
metrics.print_table(results, extra=lambda name, scores: f"{scores['wpsnr'] - baseline['wpsnr']:+8.2f} vs bicubic")

In [ ]:
"""wpsnr of every method within each group of validation patches."""


def cell(value, signed=False):
    """One number of the table, or "-" for an empty group."""
    if value is None:
        return f"{'-':>15}"
    return f"{value:+15.2f}" if signed else f"{value:15.2f}"


for key, groups in members.items():
    scores = {name: [] for name in methods}   # method -> its wpsnr in each group (None: no patches)
    for indices in groups.values():
        loader = DataLoader(Subset(val_loader.dataset, indices), batch_size=128)
        for name, method in methods.items():
            scores[name].append(training.evaluate(method, loader, trim)["wpsnr"] if indices else None)
    gains = [None if ours is None else ours - theirs for ours, theirs in zip(scores[NAME], scores["bicubic"])]

    print(f"\nwpsnr (dB) by {key}")
    print(f"{'':<16}" + "".join(f"{group:>15}" for group in groups))
    print(f"{'patches':<16}" + "".join(f"{len(indices):>15}" for indices in groups.values()))
    for name, row in scores.items():
        print(f"{name:<16}" + "".join(cell(value) for value in row))
    print(f"{'gain':<16}" + "".join(cell(value, signed=True) for value in gains))

### What it looks like

Bicubic, the model and the target. Check text lines and window edges - that is what a
desktop stream is made of.

In [ ]:
picks = [0, 400, 900, 1600]   # far apart in the set, so from different sources
lr = torch.stack([val_loader.dataset[i][0] for i in picks]).to(DEVICE)
hr = torch.stack([val_loader.dataset[i][1] for i in picks]).to(DEVICE)
with torch.no_grad():
    training.show_patches({
        "bicubic": trim(models.bicubic(lr, SCALE)),
        NAME: trim(model.eval()(lr)),
        "target": hr,
    })

## 5. Merge, save and reload

The three branches of each layer are added into one convolution. The merged model must
give the same output and the same score, because it is the one that ships. The checkpoint
holds the merged weights *and* everything needed to rebuild them; we load it into a fresh
`PlainUpscaler` and check it computes the same thing.

In [ ]:
merged = model.eval().collapsed()
training.assert_same_model(model, merged)
merged_scores = training.evaluate(merged, val_loader, trim)
assert abs(merged_scores["wpsnr"] - results[NAME]["wpsnr"]) < 0.01, "merging lost quality"
print(f"merged: {models.count_parameters(model, True):,} -> {models.count_parameters(merged, True):,} "
      f"learned parameters, wpsnr {merged_scores['wpsnr']:.2f} dB")

training.save_checkpoint(CHECKPOINT, merged, architecture="PlainUpscaler", arguments={"widths": WIDTHS, "stem": STEM},
                         trained_as=f"ReparamUpscaler(expand={EXPAND}), merged", scale=SCALE, halo=merged.halo,
                         loss="luma_chroma_l1", select="wpsnr", learning_rate=LEARNING_RATE,
                         epochs=len(history), best_epoch=best["epoch"], val=merged_scores,
                         val_bicubic=baseline, degradation=manifest["degradation"])

# rebuild the model from the file alone, the way a later notebook would
checkpoint = training.load_checkpoint(CHECKPOINT)
loaded = models.PlainUpscaler(scale=checkpoint["scale"], **checkpoint["arguments"]).to(DEVICE)
loaded.load_state_dict(checkpoint["state_dict"])
training.assert_same_model(merged, loaded)
print("reloaded model is identical")

### A whole frame

A held-out picture sent through the stream at two bitrates - the host's default, and a
tenth of it - and upscaled whole by the reloaded model. The frame edge repeats its pixels,
like a tiled client does.

In [ ]:
# the first DIV2K test picture that is at least 1920x1080, cropped to exactly that
for path in sources.files("div2k", "test"):
    picture = sources.load_image(path)
    if picture.shape[0] >= FRAME_HEIGHT and picture.shape[1] >= FRAME_WIDTH:
        picture = picture[:FRAME_HEIGHT, :FRAME_WIDTH]
        break

crop = (slice(300, 556), slice(700, 1156))   # the part of the frame shown, in HR pixels
halo = loaded.halo
fig, axes = plt.subplots(2, 3, figsize=(14, 6.4))
for row, bpp in enumerate((degrade.STREAM_BPP, degrade.STREAM_BPP / 10)):
    # a still screen after one second (30 frames): the kind of frame a stream shows most
    clip = np.repeat(picture[None], 30, axis=0)
    truth, lr = degrade.degrade(clip, degrade.Degradation(scale=SCALE, bpp=bpp))
    truth, lr = training.to_tensor(truth[-1]), training.to_tensor(lr[-1])
    with torch.no_grad():
        padded = F.pad(lr, (halo,) * 4, mode="replicate")
        outputs = {"bicubic": models.bicubic(lr, SCALE).clamp(0, 1),
                   NAME: patches.trim(loaded(padded), halo * SCALE).clamp(0, 1)}
    metrics.print_table({f"{name}, bpp {bpp:.3f}": metrics.report(image, truth) for name, image in outputs.items()})
    Image.fromarray(training.to_image(outputs[NAME])).save(training.SAMPLE_DIR / f"upscaled_reparam_bpp{bpp:.3f}.png")

    for column, (name, image) in enumerate({**outputs, "target": truth}.items()):
        axes[row][column].imshow(training.to_image(image)[crop], interpolation="nearest")
        axes[row][column].set_title(f"{name}, bpp {bpp:.3f}", fontsize=9)
        axes[row][column].axis("off")
fig.tight_layout()
plt.show()

## 6. Export for the browser

At the tile a client runs (a 320x180 step, `05_upscale_geometry.ipynb`), like notebook 7:
the model at float32, float16 (half the size), and float16 as one whole 1080p frame. Beside
it, bilinear and bicubic as one `Resize` on the same tile and frame, as the page's
baselines. Each graph is checked against torch first.

Notebooks 1-3 publish at the 128 step, so on the benchmark page compare models at the
same step.

In [ ]:
def input_shape(step_height, step_width, halo):
    """The model input for a step: the step plus the halo on every side, as (height, width)."""
    return (step_height + 2 * halo, step_width + 2 * halo)


def check_and_time(info, module, shape, dtype, tolerance):
    """Check a written graph against the torch module, time it on the CPU, print one line."""
    webexport.check_graph(info["path"], module, np.random.rand(1, 3, *shape).astype(np.float32), dtype, tolerance)
    ms = webexport.time_graph(info["path"], shape, dtype)
    webexport.annotate(info, cpu_ms=round(ms, 2))
    print(webexport.summary_line(info, ms))


tile_name = f"tile{STEP_WIDTH}x{STEP_HEIGHT}"
label = f"{NAME} (stream-degraded)"
halo = loaded.halo
tile_shape = input_shape(STEP_HEIGHT, STEP_WIDTH, halo)
frame_shape = input_shape(FRAME_HEIGHT // SCALE, FRAME_WIDTH // SCALE, halo)

fp32 = webexport.export(loaded, f"upscale_{NAME}_{tile_name}.onnx", tile_shape, label=f"{label} fp32", halo=halo)
check_and_time(fp32, loaded, tile_shape, np.float32, tolerance=2e-3)

fp16 = webexport.to_float16(fp32, f"upscale_{NAME}_fp16_{tile_name}.onnx")
webexport.write_metadata(fp16["path"], {"label": f"{label} fp16"})
check_and_time(fp16, loaded, tile_shape, np.float16, tolerance=4e-3)

frame = webexport.to_frame(fp16, f"upscale_{NAME}_fp16_{tile_name}_frame.onnx", FRAME_WIDTH, FRAME_HEIGHT)
check_and_time(frame, loaded, frame_shape, np.float16, tolerance=4e-3)

In [ ]:
"""The baselines: bilinear and bicubic as one `Resize`, on the same tile and frame.

float32 only: the float16 converter breaks `Resize`, and a filter has no weights to shrink."""

for mode, halo in (("bilinear", 1), ("bicubic", 2)):   # how far each filter reads
    module = models.Resize(SCALE, mode)
    tile_shape = input_shape(STEP_HEIGHT, STEP_WIDTH, halo)
    frame_shape = input_shape(FRAME_HEIGHT // SCALE, FRAME_WIDTH // SCALE, halo)

    tile = webexport.export(module, f"upscale_static_{mode}_{tile_name}.onnx", tile_shape,
                            label=f"static {mode} (Resize) fp32", halo=halo)
    check_and_time(tile, module, tile_shape, np.float32, tolerance=2e-3)

    frame = webexport.to_frame(tile, f"upscale_static_{mode}_{tile_name}_frame.onnx", FRAME_WIDTH, FRAME_HEIGHT)
    check_and_time(frame, module, frame_shape, np.float32, tolerance=2e-3)
print("compare in a browser: uv run src/upscale/benchmark/main.py")